Clear workspace (ak chces vsetko spustit od zaciatku, vymazat pamat notebooku, press run)

In [1]:
# Clear workspace variables (similar to MATLAB clear)

for var in list(globals().keys()):
    if not var.startswith("_"):
        del globals()[var]

print("Workspace cleared")

Workspace cleared


Import all libraries  (always run first if you dont need to clear your workplace or already did that)

In [2]:
import os
import numpy as np
import pandas as pd

Pathes of files (always run second after import LOL)

In [3]:
# Directory with raw experiment data
data_repo_dir = "../dataRepo/"
raw_dir = os.path.join(data_repo_dir, "raw")

# Directory where processed results will be stored
processed_dir = os.path.join(data_repo_dir, "processed")

# List of experiment files
experiment_files = [
    "exp1.csv",
    "exp2.csv",
    "exp3.csv",
    "exp4.csv",
    "exp5.csv"
]

# Column indices in CSV (0-based)
col_time = 0
col_u = 3
col_y1 = 1
col_y2 = 2

# Second input column index (constant input)
col_u2 = 4

# Does CSV contain header row?
csv_has_header = True

Little chcek, neni nutne

In [4]:
# Check if files exist and if CSV structure is read correctly

skip_rows = 1 if csv_has_header else 0

print("Checking files and CSV structure...\n")

for file in experiment_files:

    filepath = os.path.join(raw_dir, file)

    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        continue

    print(f"\nReading: {file}")

    try:
        data = np.loadtxt(filepath, delimiter=",", skiprows=skip_rows)

        print("Shape:", data.shape)
        print("First 3 rows:")

        preview = data[:3, [col_time, col_u, col_u2, col_y1, col_y2]]
        print(preview)

    except Exception as e:
        print("Error while reading file:", e)

Checking files and CSV structure...


Reading: exp1.csv
Shape: (36000, 5)
First 3 rows:
[[0.         0.         2.         3.11035156 3.5546875 ]
 [0.1        0.         2.         3.06152344 3.54980469]
 [0.2        0.         2.         3.11035156 3.53027344]]

Reading: exp2.csv
Shape: (36000, 5)
First 3 rows:
[[0.         0.         4.         9.66308594 7.91015625]
 [0.1        0.         4.         9.64355469 7.91015625]
 [0.2        0.         4.         9.62402344 7.72949219]]

Reading: exp3.csv
Shape: (36000, 5)
First 3 rows:
[[0.         0.         6.         9.09179688 8.33007812]
 [0.1        0.         6.         9.08203125 8.33007812]
 [0.2        0.         6.         9.05273438 8.125     ]]

Reading: exp4.csv
Shape: (36000, 5)
First 3 rows:
[[0.         0.         8.         8.53515625 7.86621094]
 [0.1        0.         8.         8.54492188 7.87109375]
 [0.2        0.         8.         8.49609375 7.65625   ]]

Reading: exp5.csv
Shape: (36000, 5)
First 3 rows:
[[ 0.   

Params of precessing of experiments

In [5]:
# Length of initial zero-input period (seconds)
zero_period_length = 600

# Length of each following period (seconds)
period_length = 300

# Number of seconds at the end of each period used for steady state calculation
steady_window = 30

# Number of last seconds of each period to exclude from processing
tail_exclude_seconds = 0.1

# Sampling time
Ts = 0.1

Extract of steady-state points

In [6]:
skip_rows = 1 if csv_has_header else 0

for file in experiment_files:

    filepath = os.path.join(raw_dir, file)
    data = np.loadtxt(filepath, delimiter=",", skiprows=skip_rows)

    time = data[:, col_time]
    u = data[:, col_u]
    u2 = data[:, col_u2]
    y1 = data[:, col_y1]
    y2 = data[:, col_y2]

    zero_samples = round(zero_period_length / Ts)
    period_samples = round(period_length / Ts)
    steady_samples = round(steady_window / Ts)
    tail_exclude_samples = round(tail_exclude_seconds / Ts)

    steady_points = []
    leakage_periods = []

    exp_name = os.path.splitext(file)[0]
    exp_folder = os.path.join(processed_dir, exp_name)
    os.makedirs(exp_folder, exist_ok=True)

    period_id = 0

    # Check if steady-state window fits after excluding tail samples
    if steady_samples + tail_exclude_samples > zero_samples:
        print(f"Error in {exp_name}: zero period is too short for selected steady window and excluded tail")
        continue

    if steady_samples + tail_exclude_samples > period_samples:
        print(f"Error in {exp_name}: period is too short for selected steady window and excluded tail")
        continue

    # Zero-input period
    if zero_samples >= steady_samples + tail_exclude_samples:
        steady_end = zero_samples - tail_exclude_samples
        steady_start = steady_end - steady_samples

        period_u = u[steady_start:steady_end]
        unique_u = np.unique(period_u)

        if len(unique_u) > 1:
            leakage_periods.append((period_id, unique_u.tolist()))

        for i in range(steady_start, steady_end):
            steady_points.append([
                period_id,
                time[i],
                u[i],
                u2[i],
                y1[i],
                y2[i]
            ])

        period_id += 1

    # All following periods
    start = zero_samples

    while start + period_samples <= len(time):
        period_end = start + period_samples

        steady_end = period_end - tail_exclude_samples
        steady_start = steady_end - steady_samples

        period_u = u[steady_start:steady_end]
        unique_u = np.unique(period_u)

        if len(unique_u) > 1:
            leakage_periods.append((period_id, unique_u.tolist()))

        for i in range(steady_start, steady_end):
            steady_points.append([
                period_id,
                time[i],
                u[i],
                u2[i],
                y1[i],
                y2[i]
            ])

        start += period_samples
        period_id += 1

    steady_points = np.array(steady_points)

    output_file = os.path.join(exp_folder, f"{exp_name}_steady_points.csv")

    np.savetxt(
        output_file,
        steady_points,
        delimiter=",",
        header="period_id,time,u,u2,y1,y2",
        comments="",
        fmt="%.5g"
    )

    print(f"Saved steady points for {exp_name}")

    if len(leakage_periods) == 0:
        print(f"No u leakage detected in selected steady-state windows of {exp_name}")
    else:
        print(f"u leakage detected in selected steady-state windows of {exp_name}:")
        for pid, values in leakage_periods:
            print(f"  period {pid}: u values = {values}")

Saved steady points for exp1
No u leakage detected in selected steady-state windows of exp1
Saved steady points for exp2
No u leakage detected in selected steady-state windows of exp2
Saved steady points for exp3
No u leakage detected in selected steady-state windows of exp3
Saved steady points for exp4
No u leakage detected in selected steady-state windows of exp4
Saved steady points for exp5
No u leakage detected in selected steady-state windows of exp5


Calculating of average values

In [7]:
for file in experiment_files:

    exp_name = os.path.splitext(file)[0]
    exp_folder = os.path.join(processed_dir, exp_name)

    input_file = os.path.join(exp_folder, f"{exp_name}_steady_points.csv")
    data = np.loadtxt(input_file, delimiter=",", skiprows=1)

    period_id = data[:, 0].astype(int)
    u = data[:, 2]
    u2 = data[:, 3]
    y1 = data[:, 4]
    y2 = data[:, 5]

    unique_periods = np.unique(period_id)

    averages = []

    for pid in unique_periods:
        mask = period_id == pid

        u_avg = np.mean(u[mask])
        u2_avg = np.mean(u2[mask])
        y1_avg = np.mean(y1[mask])
        y2_avg = np.mean(y2[mask])

        averages.append([
            pid,
            u_avg,
            u2_avg,
            y1_avg,
            y2_avg
        ])

    averages = np.array(averages)

    output_file = os.path.join(exp_folder, f"{exp_name}_steady_avg.csv")

    np.savetxt(
        output_file,
        averages,
        delimiter=",",
        header="period_id,u,u2,y1,y2",
        comments="",
        fmt="%.5g"
    )

    print(f"Saved averaged steady points for {exp_name}")

Saved averaged steady points for exp1
Saved averaged steady points for exp2
Saved averaged steady points for exp3
Saved averaged steady points for exp4
Saved averaged steady points for exp5
